## What this notebook is

A one-off check, not a deliverable.

**Label checks** - integrity, class balance, box size, class co-occurrence -
run over all three splits (train, valid, test). The drawn ground-truth samples
are from the training split.

**Prediction-error analysis** - ranking the images the model got wrong and
diagnosing each error, including whether a "hallucination" is really a missing
annotation - runs on the test split only.

Validation is what picks which of the four trained runs gets analysed, and it
supplies the training curves; Ultralytics' own val_batch previews are shown
alongside them.

**Running this writes to disk**: figures/, data_absolute.yaml, summary.txt, a
submission/ folder of CSVs and copied run artifacts, and a zip. Ultralytics
writes runs/ during training and validation. If a Roboflow export ever ships
without a test split, the setup cell moves images out of valid/ into test/.


In [ ]:
import subprocess, sys

print("=" * 78)
try:
    out = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=60)
    print(out.stdout if out.returncode == 0 else "nvidia-smi returned non-zero")
except Exception as e:
    print("nvidia-smi unavailable:", e)
print("=" * 78)
print("Python :", sys.version.split()[0])

try:
    import torch
    print("torch  :", torch.__version__)
    print("CUDA   :", torch.version.cuda, "| available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print("GPU    : %s  (%.1f GB)" % (p.name, p.total_memory / 1024**3))
    else:
        print("\n*** NO GPU DETECTED ***")
except ImportError:
    print("torch not importable")

In [ ]:
%pip install -q "ultralytics>=8.3.0" "roboflow>=1.1.30"

import ultralytics, roboflow
print("ultralytics:", ultralytics.__version__)
print("roboflow   :", roboflow.__version__)

In [ ]:
import os, time, math, random, shutil, warnings, textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import cv2
import yaml
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from PIL import Image
from IPython.display import display

warnings.filterwarnings("ignore")
os.environ["WANDB_DISABLED"] = "true"
os.environ["ULTRALYTICS_HUB"] = "false"

SEED         = 42

EPOCHS       = 30

IMGSZ        = 640
BATCH        = 16
WORKERS      = 2

CONF_DEPLOY  = 0.25
IOU_NMS      = 0.70
IOU_MATCH    = 0.50
IOU_LOC      = 0.10

RF_WORKSPACE = "first-group-project"
RF_PROJECT   = "wheelchair-9qvfx-bchvo"
RF_VERSION   = 3
RF_FORMAT    = "yolov8"

BUILD_TEST_SPLIT   = True
TEST_FRACTION      = 0.50
MIN_TEST_IMAGES    = 20

BOX_GAIN = 7.5
CLS_GAIN = 0.5
DFL_GAIN = 1.5

DISPLAY_NAME = {
    "person":            "person",
    "wheelchair":        "wheelchair (empty)",
    "people_wheelchair": "person in wheelchair",
}

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

DEVICE = 0 if torch.cuda.is_available() else "cpu"

ROOT        = Path("/content") if Path("/content").exists() else Path.cwd()
PROJECT_DIR = ROOT / "cv_project"
DATA_DIR    = PROJECT_DIR / "datasets"
RUNS_DIR    = PROJECT_DIR / "runs"
FIG_DIR     = PROJECT_DIR / "figures"
for d in (PROJECT_DIR, DATA_DIR, RUNS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def imread(path):
    img = cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise IOError("could not decode image (corrupt file?): %s" % path)
    return img

print("device      :", DEVICE, "(0 = first CUDA GPU)")
print("seed        :", SEED)
print("epochs      :", EPOCHS)
print("project dir :", PROJECT_DIR)

In [ ]:
SURFACE = "#fcfcfb"
INK     = "#0b0b0b"
INK2    = "#52514e"
MUTED   = "#898781"
GRID    = "#e1e0d9"
AXIS    = "#c3c2b7"

CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
       "#e87ba4", "#008300", "#4a3aa7", "#e34948"]

STATUS = {"good": "#0ca30c", "warning": "#fab219", "serious": "#ec835a", "critical": "#d03b3b"}

SEQ = LinearSegmentedColormap.from_list("seq_blue", [
    "#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7",
    "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "grid.color": GRID, "grid.linewidth": 0.8, "axes.grid": True, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlepad": 10,
    "legend.frameon": False, "figure.dpi": 110, "lines.linewidth": 2.0,
})

def show_fig(fig, name):
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    path = FIG_DIR / (name + ".png")
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor=SURFACE)
    plt.show()
    return path

def bar_labels(ax, bars, fmt="%d", pad=2, color=INK2, fontsize=8):
    for b in bars:
        h = b.get_height()
        if h is None or (isinstance(h, float) and math.isnan(h)):
            continue
        ax.annotate(fmt % h, (b.get_x() + b.get_width() / 2, h),
                    textcoords="offset points", xytext=(0, pad),
                    ha="center", va="bottom", fontsize=fontsize, color=color)

def dname(raw):
    return DISPLAY_NAME.get(raw, raw)

print("Plot theme ready. Figures will be saved to:", FIG_DIR)

In [ ]:
from getpass import getpass
from roboflow import Roboflow

ROBOFLOW_API_KEY = ""
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = (userdata.get("ROBOFLOW_API_KEY") or "").strip()
    if ROBOFLOW_API_KEY:
        print("Using API key from Colab Secrets.")
except Exception:
    pass

if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "").strip()

if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = getpass("Roboflow Private API key (input hidden): ").strip()

assert ROBOFLOW_API_KEY, "No API key supplied - cannot download the dataset."

rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)

if RF_VERSION is None:
    found = []
    for v in project.versions():
        try:
            found.append(int(str(v.version).rstrip("/").split("/")[-1]))
        except Exception:
            continue
    assert found, ("No generated versions found in %s/%s. Open the project in Roboflow "
                   "and press Generate." % (RF_WORKSPACE, RF_PROJECT))
    RF_VERSION = max(found)
    print("versions available: %s  ->  using newest: %d" % (sorted(found), RF_VERSION))
else:
    print("using pinned version:", RF_VERSION)

try:
    print("\nProject metadata reported by Roboflow:")
    print("  type          :", project.type)
    print("  total images  :", project.images)
    print("  classes       :", project.classes)
except Exception as e:
    print("  (metadata unavailable:", e, ")")

DL_TARGET = DATA_DIR / ("wheelchair-v%d" % RF_VERSION)
dataset = None
for fmt in dict.fromkeys([RF_FORMAT, "yolov11", "yolov8"]):
    try:
        print("\nDownloading version %d in '%s' format ..." % (RF_VERSION, fmt))
        dataset = project.version(RF_VERSION).download(fmt, location=str(DL_TARGET), overwrite=True)
        print("OK  ->", dataset.location)
        break
    except Exception as e:
        print("  '%s' failed: %s" % (fmt, e))

assert dataset is not None, "Download failed in every format. Check the API key and your network."
DS_ROOT = Path(dataset.location)

In [ ]:
def find_split_dirs(root: Path):
    found = {}
    for key, candidates in [("train", ["train"]), ("val", ["valid", "val"]), ("test", ["test"])]:
        for c in candidates:
            if (root / c / "images").is_dir():
                found[key] = root / c
                break
    return found

splits = find_split_dirs(DS_ROOT)
def n_imgs(p):
    return len([f for f in (p / "images").iterdir() if f.suffix.lower() in IMG_EXT]) if p else 0

print("As downloaded:")
for k in ("train", "val", "test"):
    print("  %-5s : %s" % (k, ("%4d images" % n_imgs(splits[k])) if k in splits else "MISSING"))

need_test = BUILD_TEST_SPLIT and n_imgs(splits.get("test")) < MIN_TEST_IMAGES
TEST_SPLIT_IS_NATIVE = not need_test

if need_test:
    src = splits["val"]
    dst = DS_ROOT / "test"
    (dst / "images").mkdir(parents=True, exist_ok=True)
    (dst / "labels").mkdir(parents=True, exist_ok=True)

    imgs = sorted([f for f in (src / "images").iterdir() if f.suffix.lower() in IMG_EXT])
    rng = random.Random(SEED)
    rng.shuffle(imgs)
    n_move = int(round(len(imgs) * TEST_FRACTION))
    moved = 0
    for ip in imgs[:n_move]:
        lp = src / "labels" / (ip.stem + ".txt")
        shutil.move(str(ip), str(dst / "images" / ip.name))
        if lp.exists():
            shutil.move(str(lp), str(dst / "labels" / lp.name))
        moved += 1

    print("  moved %d of %d validation images into test/ (seed=%d, fraction=%.2f)"
          % (moved, len(imgs), SEED, TEST_FRACTION))
    print("  Deterministic: re-running gives the identical split.")
    splits = find_split_dirs(DS_ROOT)
else:
    print("\nThe export ships its own test split (%d images) - left untouched."
          % n_imgs(splits.get("test")))

for _k in ("train", "val", "test"):
    assert _k in splits, (
        "No '%s' split found under %s. Expected Roboflow's train/ valid/ test/ layout, each "
        "with images/ and labels/. If you set BUILD_TEST_SPLIT=False, set it back to True."
        % (_k, DS_ROOT))

raw_yaml = yaml.safe_load((DS_ROOT / "data.yaml").read_text(encoding="utf-8"))
names = raw_yaml.get("names")
if isinstance(names, dict):
    names = [names[k] for k in sorted(names, key=lambda x: int(x))]
CLASS_NAMES = list(names)
NC = len(CLASS_NAMES)

DATA_YAML = DS_ROOT / "data_absolute.yaml"
cfg = {
    "path":  str(DS_ROOT),
    "train": str(splits["train"] / "images"),
    "val":   str(splits["val"]   / "images"),
    "test":  str(splits["test"]  / "images"),
    "nc":    NC,
    "names": CLASS_NAMES,
}
DATA_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
TEST_SPLIT = "test"

test_images = sorted([p for p in (splits[TEST_SPLIT] / "images").iterdir()
                      if p.suffix.lower() in IMG_EXT])

_tot = sum(n_imgs(splits[k]) for k in ("train", "val", "test"))
for k in ("train", "val", "test"):
    n = n_imgs(splits[k])
    print("  %-5s : %4d images  (%.1f%%)" % (k, n, 100 * n / max(_tot, 1)))
print("  %-5s : %4d images" % ("TOTAL", _tot))
print("\nClasses (%d): %s" % (NC, CLASS_NAMES))
print("Display names:", [dname(c) for c in CLASS_NAMES])

for i, c in enumerate(CLASS_NAMES):
    print("  %d -> %s" % (i, c))

expected = {"person", "wheelchair", "people_wheelchair"}
if set(CLASS_NAMES) != expected:
    print("\n!! WARNING: class NAMES differ from the expected %s." % sorted(expected))

print("\ndata_absolute.yaml:")
print(DATA_YAML.read_text(encoding="utf-8"))

In [ ]:
def check_split(split_dir: Path, nc: int):
    img_dir, lbl_dir = split_dir / "images", split_dir / "labels"
    imgs = sorted([f for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXT])
    lbls = sorted(lbl_dir.glob("*.txt")) if lbl_dir.is_dir() else []
    img_stems, lbl_stems = {f.stem for f in imgs}, {f.stem for f in lbls}

    report = {
        "images": len(imgs),
        "label_files": len(lbls),
        "images_without_label_file": len(img_stems - lbl_stems),
        "orphan_labels": len(lbl_stems - img_stems),
        "empty_label_files": 0, "malformed_lines": 0,
        "bad_class_id": 0, "coords_out_of_range": 0, "degenerate_boxes": 0, "boxes": 0,
    }
    for lp in lbls:
        lines = [ln for ln in lp.read_text(encoding="utf-8").strip().splitlines() if ln.strip()]
        if not lines:
            report["empty_label_files"] += 1
            continue
        for ln in lines:
            p = ln.split()
            if len(p) < 5 or (len(p) > 5 and (len(p) - 1) % 2 != 0):
                report["malformed_lines"] += 1
                continue
            try:
                c = int(float(p[0])); vals = [float(v) for v in p[1:]]
            except ValueError:
                report["malformed_lines"] += 1
                continue
            report["boxes"] += 1
            if not (0 <= c < nc):
                report["bad_class_id"] += 1
            if len(p) == 5:
                cx, cy, w, h = vals
                if not all(-0.001 <= v <= 1.001 for v in (cx, cy, w, h)):
                    report["coords_out_of_range"] += 1
                if w <= 0 or h <= 0:
                    report["degenerate_boxes"] += 1
    return report

rows = {k: check_split(splits[k], NC) for k in ("train", "val", "test")}
integrity = pd.DataFrame(rows).T
display(integrity)

problems = ["orphan_labels", "malformed_lines", "bad_class_id",
            "coords_out_of_range", "degenerate_boxes"]
total_bad = int(integrity[problems].values.sum())
if total_bad == 0:
    print("PASS - no structural problems found.")
else:
    print("FOUND %d structural problem(s). Inspect the columns above before training." % total_bad)

print("\nimages %d | label files %d | boxes %d   (all counted from disk)"
      % (int(integrity["images"].sum()), int(integrity["label_files"].sum()),
         int(integrity["boxes"].sum())))

bg = int(integrity["images_without_label_file"].sum())
print("\n%d image(s) have no label file (background images)." % bg)

In [ ]:
def parse_split(split_key: str):
    split_dir = splits[split_key]
    ann_rows, img_rows, unreadable = [], [], []
    for ip in sorted([f for f in (split_dir / "images").iterdir() if f.suffix.lower() in IMG_EXT]):
        try:
            with Image.open(ip) as _im:
                W, H = _im.size
        except Exception as e:
            unreadable.append((split_key, ip.name, "%s: %s" % (type(e).__name__, e)))
            continue
        lp = split_dir / "labels" / (ip.stem + ".txt")
        n = 0
        if lp.exists():
            for ln in lp.read_text(encoding="utf-8").strip().splitlines():
                p = ln.split()
                if len(p) < 5:
                    continue
                try:
                    c = int(float(p[0])); vals = [float(v) for v in p[1:]]
                except ValueError:
                    continue
                if len(p) == 5:
                    cx, cy, w, h = vals
                else:
                    xs, ys = vals[0::2], vals[1::2]
                    cx, cy = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2
                    w,  h  = max(xs) - min(xs),      max(ys) - min(ys)
                if w <= 0 or h <= 0 or not (0 <= c < NC):
                    continue
                ann_rows.append(dict(
                    split=split_key, image=ip.name, cls_id=c, cls=CLASS_NAMES[c],
                    cx=cx, cy=cy, w=w, h=h,
                    rel_area=w * h,
                    px_area=(w * IMGSZ) * (h * IMGSZ),
                    aspect=w / h if h > 0 else np.nan))
                n += 1
        img_rows.append(dict(split=split_key, image=ip.name, W=W, H=H, n_boxes=n))
    return ann_rows, img_rows, unreadable

_a, _i, _bad = [], [], []
for k in ("train", "val", "test"):
    a, i, b = parse_split(k)
    _a += a; _i += i; _bad += b
ann_df = pd.DataFrame(_a)
img_df = pd.DataFrame(_i)

if _bad:
    print("!! %d image(s) could not be opened and are EXCLUDED from every number below:" % len(_bad))
    for sp, nm, err in _bad[:15]:
        print("     [%s] %s  -  %s" % (sp, nm, err))
    if len(_bad) > 15:
        print("     ... and %d more" % (len(_bad) - 15))
else:
    print("All images opened cleanly.\n")

SPLIT_ORDER = ["train", "val", "test"]
ann_df["split"] = pd.Categorical(ann_df["split"], SPLIT_ORDER, ordered=True)
img_df["split"] = pd.Categorical(img_df["split"], SPLIT_ORDER, ordered=True)

print("Parsed %d boxes across %d images.\n" % (len(ann_df), len(img_df)))
summary = pd.DataFrame({
    "images":          img_df.groupby("split", observed=True).size(),
    "boxes":           ann_df.groupby("split", observed=True).size(),
    "boxes_per_image": ann_df.groupby("split", observed=True).size() /
                       img_df.groupby("split", observed=True).size(),
    "empty_images":    img_df[img_df.n_boxes == 0].groupby("split", observed=True).size(),
}).fillna(0).round(2)
display(summary)

print("TOTAL (measured on disk): %d images, %d boxes"
      % (len(img_df), len(ann_df)))
print("Split proportions       : " + " / ".join(
    "%s %.1f%%" % (s, 100 * (img_df.split == s).sum() / max(len(img_df), 1)) for s in SPLIT_ORDER))

res = img_df[["W", "H"]].drop_duplicates().values.tolist()
print("\nUnique image resolutions on disk:", res[:5], "" if len(res) <= 5 else "(+%d more)" % (len(res) - 5))

In [ ]:
CLASS_COUNTS = (ann_df.groupby(["cls", "split"], observed=True).size()
                .unstack(fill_value=0).reindex(columns=SPLIT_ORDER, fill_value=0))
CLASS_COUNTS = CLASS_COUNTS.loc[CLASS_COUNTS.sum(axis=1).sort_values(ascending=False).index]
CLASS_COUNTS["total"] = CLASS_COUNTS.sum(axis=1)
display(CLASS_COUNTS)

cls_order = list(CLASS_COUNTS.index)

CLASS_COLOR   = {c: CAT[i % len(CAT)] for i, c in enumerate(cls_order)}
DISPLAY_COLOR = {dname(c): v for c, v in CLASS_COLOR.items()}

SPLIT_COLOR = {"train": "#184f95", "val": "#3987e5", "test": "#9ec5f4"}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

ax = axes[0]
x = np.arange(len(cls_order)); width = 0.26
for j, sp in enumerate(SPLIT_ORDER):
    bars = ax.bar(x + (j - 1) * width, CLASS_COUNTS[sp].values, width,
                  label=sp, color=SPLIT_COLOR[sp], edgecolor=SURFACE, linewidth=1.5)
    bar_labels(ax, bars)
ax.set_xticks(x); ax.set_xticklabels([dname(c) for c in cls_order])
ax.set_ylabel("annotated boxes"); ax.set_title("Class distribution by split")
ax.legend(ncol=3, loc="upper right"); ax.grid(axis="x", visible=False)

ax = axes[1]
tr = CLASS_COUNTS["train"]; share = 100 * tr / max(tr.sum(), 1)
bars = ax.bar([dname(c) for c in cls_order], share.values,
              color=[CLASS_COLOR[c] for c in cls_order], edgecolor=SURFACE,
              linewidth=1.5, width=0.6)
bar_labels(ax, bars, fmt="%.1f%%")
ax.set_ylabel("% of training boxes"); ax.set_title("Training-set class share")
ax.set_ylim(0, max(share.max() * 1.18, 10)); ax.grid(axis="x", visible=False)

show_fig(fig, "01_class_distribution")

imb = tr.max() / max(tr.min(), 1)
print("Imbalance ratio (most vs least frequent class, train): %.1f : 1" % imb)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.3))

ax = axes[0]
vals = img_df.n_boxes.values
ax.hist(vals, bins=np.arange(0, min(vals.max(), 25) + 2) - 0.5,
        color=CAT[0], edgecolor=SURFACE, linewidth=1.2)
ax.set_xlabel("objects in image"); ax.set_ylabel("images")
ax.set_title("Objects per image")
ax.axvline(vals.mean(), color=STATUS["critical"], lw=2, ls="--")
ax.annotate("mean %.1f" % vals.mean(), (vals.mean(), ax.get_ylim()[1] * 0.9),
            xytext=(6, 0), textcoords="offset points", color=STATUS["critical"], fontsize=9)
ax.grid(axis="x", visible=False)

ax = axes[1]
for c in cls_order:
    s = np.sqrt(ann_df.loc[ann_df.cls == c, "px_area"].values)
    ax.hist(s, bins=40, histtype="step", lw=2, color=CLASS_COLOR[c], label=dname(c))
for edge, lab in [(32, "small|medium"), (96, "medium|large")]:
    ax.axvline(edge, color=AXIS, lw=1.5, ls=":")
    ax.annotate(lab, (edge, ax.get_ylim()[1] * 0.96), rotation=90, fontsize=7.5,
                color=MUTED, ha="right", va="top")
ax.set_xlabel("sqrt(box area) in pixels @ 640"); ax.set_ylabel("boxes")
ax.set_title("Object size (COCO bands)"); ax.legend(); ax.grid(axis="x", visible=False)

ax = axes[2]
for c in cls_order:
    d = ann_df[ann_df.cls == c]
    ax.scatter(d.w * IMGSZ, d.h * IMGSZ, s=7, alpha=0.35, color=CLASS_COLOR[c],
               label=dname(c), edgecolors="none")
lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([0, lim], [0, lim], color=AXIS, lw=1.2, ls=":")
ax.annotate("square", (lim * 0.82, lim * 0.86), color=MUTED, fontsize=8, rotation=45)
ax.set_xlabel("box width (px @640)"); ax.set_ylabel("box height (px @640)")
ax.set_title("Box shape by class")
ax.legend(markerscale=2.2)

show_fig(fig, "02_object_geometry")

sz = pd.DataFrame({
    "median sqrt(area) px": ann_df.groupby("cls", observed=True).px_area.median().pow(0.5).round(1),
    "median aspect w/h":    ann_df.groupby("cls", observed=True).aspect.median().round(2),
    "% small (<32px)":      ann_df.groupby("cls", observed=True).px_area
                              .apply(lambda s: 100 * (s < 32**2).mean()).round(1),
    "% large (>96px)":      ann_df.groupby("cls", observed=True).px_area
                              .apply(lambda s: 100 * (s > 96**2).mean()).round(1),
}).reindex(cls_order)
sz.index = [dname(c) for c in sz.index]
display(sz)

In [ ]:
fig, axes = plt.subplots(1, len(cls_order), figsize=(4.6 * len(cls_order), 4.3))
axes = np.atleast_1d(axes)
for i, c in enumerate(cls_order):
    ax = axes[i]
    d = ann_df[ann_df.cls == c]
    hb = ax.hist2d(d.cx.values, d.cy.values, bins=28, range=[[0, 1], [0, 1]], cmap=SEQ)
    ax.set_title("%s  (n=%d)" % (dname(c), len(d)))
    ax.set_xlabel("x (normalised)"); ax.set_ylabel("y (normalised)" if i == 0 else "")
    ax.invert_yaxis()
    ax.grid(False)
    ax.set_aspect("equal")
    plt.colorbar(hb[3], ax=ax, fraction=0.046, label="boxes" if i == len(cls_order) - 1 else "")
fig.suptitle("Spatial distribution of box centres", x=0.09, ha="left", fontsize=12, fontweight="bold")
show_fig(fig, "03_spatial_prior")

In [ ]:
per_img = (ann_df.groupby(["split", "image"], observed=True)["cls"]
           .apply(lambda s: set(s)).reset_index(name="classes"))
co = pd.DataFrame(0, index=cls_order, columns=cls_order, dtype=int)
for cs in per_img["classes"]:
    for a in cs:
        for b in cs:
            if a in co.index and b in co.columns:
                co.loc[a, b] += 1

fig, ax = plt.subplots(figsize=(6.2, 5.2))
im = ax.imshow(co.values, cmap=SEQ)
ax.set_xticks(range(len(cls_order))); ax.set_xticklabels([dname(c) for c in cls_order],
                                                        rotation=20, ha="right")
ax.set_yticks(range(len(cls_order))); ax.set_yticklabels([dname(c) for c in cls_order])
vmax = co.values.max()
for i in range(len(cls_order)):
    for j in range(len(cls_order)):
        v = co.values[i, j]
        ax.text(j, i, str(v), ha="center", va="center", fontsize=10,
                color="#ffffff" if v > 0.55 * vmax else INK)
ax.set_title("Images containing both classes (diagonal = images containing the class)")
ax.grid(False)
plt.colorbar(im, ax=ax, fraction=0.046, label="images")
show_fig(fig, "04_class_cooccurrence")

n_both = co.loc["person", "people_wheelchair"] if {"person", "people_wheelchair"} <= set(co.index) else 0
print("Images containing BOTH 'person' and 'people_wheelchair': %d" % n_both)

In [ ]:
def hex2bgr(h):
    h = h.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return (b, g, r)

def draw_boxes(img_bgr, boxes, labels, colors, thickness=2, font_scale=0.5):
    out = img_bgr.copy()
    for (x1, y1, x2, y2), lab, col in zip(boxes, labels, colors):
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        cv2.rectangle(out, (x1, y1), (x2, y2), col, thickness)
        (tw, th), base = cv2.getTextSize(lab, cv2.FONT_HERSHEY_SIMPLEX, font_scale, 1)
        ytop = max(0, y1 - th - base - 2)
        cv2.rectangle(out, (x1, ytop), (x1 + tw + 4, ytop + th + base + 2), col, -1)
        cv2.putText(out, lab, (x1 + 2, ytop + th + 1),
                    cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), 1, cv2.LINE_AA)
    return out

def yolo_to_xyxy(cx, cy, w, h, W, H):
    return ((cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H)

def load_gt(split_key, image_name):
    ip = splits[split_key] / "images" / image_name
    img = imread(ip)
    H, W = img.shape[:2]
    boxes, ids = [], []
    lp = splits[split_key] / "labels" / (Path(image_name).stem + ".txt")
    if lp.exists():
        for ln in lp.read_text(encoding="utf-8").strip().splitlines():
            p = ln.split()
            if len(p) < 5:
                continue
            c = int(float(p[0])); vals = [float(v) for v in p[1:]]
            if len(p) == 5:
                cx, cy, w, h = vals
            else:
                xs, ys = vals[0::2], vals[1::2]
                cx, cy = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2
                w,  h  = max(xs) - min(xs),      max(ys) - min(ys)
            boxes.append(yolo_to_xyxy(cx, cy, w, h, W, H)); ids.append(c)
    return img, boxes, ids

rng = random.Random(SEED)
picked = []
for c in cls_order:
    pool = ann_df[(ann_df.split == "train") & (ann_df.cls == c)].image.unique().tolist()
    if pool:
        picked += rng.sample(pool, min(2, len(pool)))
pool_all = img_df[(img_df.split == "train") & (img_df.n_boxes > 0)].image.tolist()
while len(picked) < 9 and pool_all:
    cand = rng.choice(pool_all)
    if cand not in picked:
        picked.append(cand)
picked = picked[:9]

fig, axes = plt.subplots(3, 3, figsize=(13.5, 13.5))
for ax, name in zip(axes.ravel(), picked):
    img, boxes, ids = load_gt("train", name)
    vis = draw_boxes(img, boxes,
                     [dname(CLASS_NAMES[i]) for i in ids],
                     [hex2bgr(CLASS_COLOR[CLASS_NAMES[i]]) for i in ids])
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(name[:34], fontsize=8, color=MUTED, fontweight="normal")
    ax.axis("off")
for ax in axes.ravel()[len(picked):]:
    ax.axis("off")
handles = [Patch(facecolor=CLASS_COLOR[c], label=dname(c)) for c in cls_order]
fig.legend(handles=handles, loc="lower center", ncol=len(cls_order), bbox_to_anchor=(0.5, -0.01))
fig.suptitle("Ground-truth annotations (training set)", x=0.09, ha="left",
             fontsize=13, fontweight="bold")
show_fig(fig, "05_ground_truth_samples")

In [ ]:
def warp_boxes(boxes, M, W, H, min_area=16):
    if len(boxes) == 0:
        return np.zeros((0, 4), np.float32), np.zeros(0, bool)
    b = np.asarray(boxes, np.float32)
    n = len(b)
    corners = np.ones((n * 4, 3), np.float32)
    corners[:, :2] = b[:, [0, 1, 2, 3, 0, 3, 2, 1]].reshape(n * 4, 2)
    corners = corners @ M.T
    corners = corners.reshape(n, 4, 2)
    new = np.concatenate([corners.min(1), corners.max(1)], axis=1)
    new[:, [0, 2]] = new[:, [0, 2]].clip(0, W)
    new[:, [1, 3]] = new[:, [1, 3]].clip(0, H)
    area = (new[:, 2] - new[:, 0]) * (new[:, 3] - new[:, 1])
    keep = area > min_area
    return new, keep

def aug_hflip(img, boxes, ids):
    H, W = img.shape[:2]
    out = np.ascontiguousarray(img[:, ::-1])
    b = np.asarray(boxes, np.float32).copy()
    if len(b):
        x1 = b[:, 0].copy()
        b[:, 0] = W - b[:, 2]
        b[:, 2] = W - x1
    return out, b, list(ids)

def aug_rotate(img, boxes, ids, deg=12.0):
    H, W = img.shape[:2]
    M = cv2.getRotationMatrix2D((W / 2, H / 2), deg, 1.0)
    out = cv2.warpAffine(img, M, (W, H), borderValue=(114, 114, 114))
    nb, keep = warp_boxes(boxes, M, W, H)
    return out, nb[keep], [i for i, k in zip(ids, keep) if k]

def aug_hsv(img, boxes, ids, h_gain=0.015, s_gain=0.7, v_gain=0.4):
    r = np.random.uniform(-1, 1, 3) * np.array([h_gain, s_gain, v_gain]) + 1
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.int16)
    hsv[..., 0] = (hsv[..., 0] * r[0]) % 180
    hsv[..., 1] = np.clip(hsv[..., 1] * r[1], 0, 255)
    hsv[..., 2] = np.clip(hsv[..., 2] * r[2], 0, 255)
    out = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    return out, np.asarray(boxes, np.float32), list(ids)

def aug_scale_translate(img, boxes, ids, scale=0.6, tx=0.12, ty=-0.08):
    H, W = img.shape[:2]
    M = np.array([[scale, 0, tx * W + (1 - scale) * W / 2],
                  [0, scale, ty * H + (1 - scale) * H / 2]], np.float32)
    out = cv2.warpAffine(img, M, (W, H), borderValue=(114, 114, 114))
    nb, keep = warp_boxes(boxes, M, W, H)
    return out, nb[keep], [i for i, k in zip(ids, keep) if k]

def aug_mosaic(items, out_size=640):
    s = out_size
    canvas = np.full((2 * s, 2 * s, 3), 114, np.uint8)
    all_b, all_i = [], []
    for q, (img, boxes, ids) in enumerate(items[:4]):
        H, W = img.shape[:2]
        r = cv2.resize(img, (s, s))
        ox, oy = (q % 2) * s, (q // 2) * s
        canvas[oy:oy + s, ox:ox + s] = r
        sx, sy = s / W, s / H
        for (x1, y1, x2, y2), c in zip(boxes, ids):
            all_b.append([x1 * sx + ox, y1 * sy + oy, x2 * sx + ox, y2 * sy + oy])
            all_i.append(c)
    cx = np.random.randint(int(0.35 * s), int(1.65 * s) - s + 1)
    cy = np.random.randint(int(0.35 * s), int(1.65 * s) - s + 1)
    crop = canvas[cy:cy + s, cx:cx + s].copy()
    if all_b:
        b = np.asarray(all_b, np.float32)
        b[:, [0, 2]] -= cx; b[:, [1, 3]] -= cy
        b[:, [0, 2]] = b[:, [0, 2]].clip(0, s); b[:, [1, 3]] = b[:, [1, 3]].clip(0, s)
        area = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
        keep = area > 64
        return crop, b[keep], [c for c, k in zip(all_i, keep) if k]
    return crop, np.zeros((0, 4), np.float32), []

np.random.seed(SEED)
base_name = picked[0]
img0, box0, id0 = load_gt("train", base_name)
extra_items = [load_gt("train", n) for n in picked[1:4]]

variants = [
    ("original",                    (img0, np.asarray(box0, np.float32), id0)),
    ("horizontal flip (p=0.5)",     aug_hflip(img0, box0, id0)),
    ("rotation +12 deg",            aug_rotate(img0, box0, id0, 12.0)),
    ("HSV jitter (brightness etc.)", aug_hsv(img0, box0, id0)),
    ("scale 0.6 + translate",       aug_scale_translate(img0, box0, id0)),
    ("mosaic (4 images -> 1)",      aug_mosaic([(img0, box0, id0)] + extra_items)),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10.5))
for ax, (title, (im, bx, ii)) in zip(axes.ravel(), variants):
    vis = draw_boxes(im, bx, [dname(CLASS_NAMES[c]) for c in ii],
                     [hex2bgr(CLASS_COLOR[CLASS_NAMES[c]]) for c in ii])
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title("%s   -   %d box(es)" % (title, len(bx)), fontsize=10)
    ax.axis("off")
fig.suptitle("Augmentation, implemented by hand - note the boxes follow the pixels",
             x=0.09, ha="left", fontsize=13, fontweight="bold")
show_fig(fig, "06_augmentation_demo")

In [ ]:
AUG = dict(
    hsv_h=0.015,
    hsv_s=0.70,
    hsv_v=0.40,

    degrees=10.0,
    translate=0.10,
    scale=0.50,
    shear=0.0,
    perspective=0.0,
    fliplr=0.5,
    flipud=0.0,

    mosaic=1.0,
    mixup=0.0,
    copy_paste=0.0,

    close_mosaic=0,
)

display(pd.DataFrame({"value": AUG}).T)

In [ ]:
from ultralytics import YOLO

model_probe = YOLO("yolo11n.pt")

print("=" * 78)
model_probe.info(detailed=False, verbose=True)
print("=" * 78)

seq = model_probe.model.model
rows = []
for i, m in enumerate(seq):
    rows.append(dict(idx=i,
                     module=type(m).__name__,
                     params=sum(p.numel() for p in m.parameters()),
                     from_layer=str(getattr(m, "f", ""))))
arch = pd.DataFrame(rows)
N_BACKBONE = 11
arch["part"] = np.where(arch.idx < N_BACKBONE, "backbone", "neck + head")
arch["frozen_if_freeze10"] = arch.idx < 10

display(arch)
print("Total modules      : {:d}".format(len(seq)))
print("Backbone params    : {:,}".format(int(arch.loc[arch.part == "backbone", "params"].sum())))
print("Neck+head params   : {:,}".format(int(arch.loc[arch.part != "backbone", "params"].sum())))
print("Total params       : {:,}".format(int(arch["params"].sum())))

In [ ]:
def freeze_report(model, freeze_n):
    prefixes = ["model.%d." % i for i in range(freeze_n)]
    frozen = trainable = 0
    for name, p in model.model.named_parameters():
        if any(name.startswith(pre) for pre in prefixes):
            frozen += p.numel()
        else:
            trainable += p.numel()
    total = frozen + trainable
    return dict(freeze=freeze_n, frozen=frozen, trainable=trainable, total=total,
                pct_trainable=round(100 * trainable / total, 1))

tbl = pd.DataFrame([freeze_report(model_probe, n) for n in (0, 10, 11, len(seq) - 1)])
tbl["meaning"] = ["full fine-tuning (everything trains)",
                  "backbone frozen -> feature extraction",
                  "backbone + first neck layer frozen",
                  "only the detection head trains"]
display(tbl)

del model_probe
torch.cuda.empty_cache()

In [ ]:
RUN_REGISTRY = {}

WARMUP_EPOCHS = 3.0 if EPOCHS >= 10 else 1.0

def train_run(tag, weights, epochs=None, freeze=0, lr0=0.01, batch=None, imgsz=None, **overrides):
    epochs = EPOCHS if epochs is None else epochs
    batch  = BATCH  if batch  is None else batch
    imgsz  = IMGSZ  if imgsz  is None else imgsz

    set_seed(SEED)
    model = YOLO(weights)

    kw = dict(
        data=str(DATA_YAML), epochs=epochs, imgsz=imgsz, batch=batch,
        device=DEVICE, workers=WORKERS, seed=SEED, deterministic=True,
        project=str(RUNS_DIR), name=tag, exist_ok=True,

        optimizer="SGD", lr0=lr0, lrf=0.01, momentum=0.937, weight_decay=5e-4,
        warmup_epochs=WARMUP_EPOCHS, warmup_momentum=0.8, cos_lr=False,

        box=BOX_GAIN, cls=CLS_GAIN, dfl=DFL_GAIN,

        freeze=freeze, pretrained=True,

        patience=0, val=True, plots=True, save=True, verbose=True, cache=False,
    )
    kw.update(AUG)
    kw.update(overrides)

    banner = "  RUN %s  |  %s  |  freeze=%d  lr0=%g  batch=%d  epochs=%d  " % (
        tag, weights, freeze, lr0, batch, epochs)
    print("\n" + "=" * len(banner)); print(banner); print("=" * len(banner))

    t0 = time.perf_counter()
    try:
        model.train(**kw)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise RuntimeError("CUDA out of memory. Lower BATCH (16 -> 8) or IMGSZ (640 -> 512) "
                           "in the config cell and re-run.")
    minutes = (time.perf_counter() - t0) / 60

    save_dir = Path(getattr(getattr(model, "trainer", None), "save_dir", RUNS_DIR / tag))
    RUN_REGISTRY[tag] = dict(
        tag=tag, weights=weights, epochs=epochs, freeze=freeze, lr0=lr0, batch=batch,
        save_dir=str(save_dir), best=str(save_dir / "weights" / "best.pt"),
        last=str(save_dir / "weights" / "last.pt"), train_minutes=round(minutes, 2))

    print("\nfinished %s in %.2f min  ->  %s" % (tag, minutes, save_dir))
    del model
    torch.cuda.empty_cache()
    return save_dir

print("epochs = %d   warmup_epochs = %.1f   loss gains: box=%.1f cls=%.1f dfl=%.1f"
      % (EPOCHS, WARMUP_EPOCHS, BOX_GAIN, CLS_GAIN, DFL_GAIN))

In [ ]:
_ = train_run("A_frozen_lr01", "yolo11n.pt", freeze=10, lr0=0.01)

In [ ]:
_ = train_run("B_full_lr01", "yolo11n.pt", freeze=0, lr0=0.01)

In [ ]:
_ = train_run("C_full_lr001", "yolo11n.pt", freeze=0, lr0=0.001)

In [ ]:
_ = train_run("D_yolo11s_lr01", "yolo11s.pt", freeze=0, lr0=0.01)

print("\n\nAll runs complete.")
display(pd.DataFrame(RUN_REGISTRY).T[["weights", "freeze", "lr0", "epochs", "train_minutes"]])

In [ ]:
def read_results(tag):
    p = Path(RUN_REGISTRY[tag]["save_dir"]) / "results.csv"
    if not p.exists():
        print("!! missing", p); return None
    df = pd.read_csv(p)
    df.columns = [c.strip() for c in df.columns]
    return df

def rcol(df, *cands):
    for c in cands:
        if c in df.columns:
            return df[c].values
    return None

HIST = {t: read_results(t) for t in RUN_REGISTRY}
HIST = {t: d for t, d in HIST.items() if d is not None}
TAGS = list(HIST.keys())
print("Loaded histories for:", TAGS)
print("Columns available   :", list(HIST[TAGS[0]].columns))
display(HIST[TAGS[0]].round(4))

In [ ]:
PANELS = [
    ("box_loss",  ["train/box_loss"], ["val/box_loss"], "CIoU box loss"),
    ("cls_loss",  ["train/cls_loss"], ["val/cls_loss"], "classification loss"),
    ("dfl_loss",  ["train/dfl_loss"], ["val/dfl_loss"], "DFL loss"),
]

def plot_run(tag):
    df = HIST[tag]
    ep = rcol(df, "epoch")
    fig, axes = plt.subplots(2, 3, figsize=(15.5, 8))

    for ax, (key, tr_c, va_c, nice) in zip(axes[0], PANELS):
        tr, va = rcol(df, *tr_c), rcol(df, *va_c)
        if tr is not None:
            ax.plot(ep, tr, color=CAT[0], marker="o", ms=4, label="train")
        if va is not None:
            ax.plot(ep, va, color=CAT[1], marker="s", ms=4, label="validation")
        ax.set_title(nice); ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend()

    ax = axes[1][0]
    for i, (c, lab) in enumerate([("metrics/mAP50(B)", "mAP@0.5"),
                                  ("metrics/mAP50-95(B)", "mAP@0.5:0.95")]):
        v = rcol(df, c)
        if v is not None:
            ax.plot(ep, v, color=CAT[i], marker="o", ms=4, label=lab)
    ax.set_title("validation mAP"); ax.set_xlabel("epoch"); ax.set_ylabel("mAP")
    ax.set_ylim(0, 1); ax.legend()

    ax = axes[1][1]
    for i, (c, lab) in enumerate([("metrics/precision(B)", "precision"),
                                  ("metrics/recall(B)", "recall")]):
        v = rcol(df, c)
        if v is not None:
            ax.plot(ep, v, color=CAT[i], marker="o", ms=4, label=lab)
    ax.set_title("validation precision / recall"); ax.set_xlabel("epoch")
    ax.set_ylabel("score"); ax.set_ylim(0, 1); ax.legend()

    ax = axes[1][2]
    lr = rcol(df, "lr/pg0")
    if lr is not None:
        ax.plot(ep, lr, color=CAT[2], marker="o", ms=4)
        ax.set_title("learning rate (param group 0)")
        ax.set_xlabel("epoch"); ax.set_ylabel("lr")
        ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    meta = RUN_REGISTRY[tag]
    fig.suptitle("%s   -   freeze=%d, lr0=%g, %s" % (tag, meta["freeze"], meta["lr0"],
                                                     meta["weights"]),
                 x=0.09, ha="left", fontsize=13, fontweight="bold")
    show_fig(fig, "07_curves_%s" % tag)

for t in TAGS:
    plot_run(t)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

ax = axes[0]
for i, t in enumerate(TAGS):
    df = HIST[t]
    v = rcol(df, "metrics/mAP50-95(B)")
    if v is not None:
        ax.plot(rcol(df, "epoch"), v, color=CAT[i], marker="o", ms=4, label=t)
        ax.annotate(t.split("_")[0], (rcol(df, "epoch")[-1], v[-1]), xytext=(5, 0),
                    textcoords="offset points", color=CAT[i], fontsize=9, va="center")
ax.set_title("Validation mAP@0.5:0.95"); ax.set_xlabel("epoch"); ax.set_ylabel("mAP")
ax.set_ylim(bottom=0); ax.legend(fontsize=8)

ax = axes[1]
for i, t in enumerate(TAGS):
    df = HIST[t]
    v = rcol(df, "val/box_loss")
    if v is not None:
        ax.plot(rcol(df, "epoch"), v, color=CAT[i], marker="o", ms=4, label=t)
ax.set_title("Validation box loss (lower is better)"); ax.set_xlabel("epoch")
ax.set_ylabel("CIoU loss"); ax.legend(fontsize=8)

ax = axes[2]
finals = []
for t in TAGS:
    df = HIST[t]
    v50, v95 = rcol(df, "metrics/mAP50(B)"), rcol(df, "metrics/mAP50-95(B)")
    finals.append((t, float(v50[-1]) if v50 is not None else np.nan,
                      float(v95[-1]) if v95 is not None else np.nan))
x = np.arange(len(finals)); width = 0.38
b1 = ax.bar(x - width / 2, [f[1] for f in finals], width, label="mAP@0.5",
            color=CAT[0], edgecolor=SURFACE, linewidth=1.5)
b2 = ax.bar(x + width / 2, [f[2] for f in finals], width, label="mAP@0.5:0.95",
            color=CAT[1], edgecolor=SURFACE, linewidth=1.5)
bar_labels(ax, b1, fmt="%.3f"); bar_labels(ax, b2, fmt="%.3f")
ax.set_xticks(x); ax.set_xticklabels([f[0].split("_")[0] for f in finals])
ax.set_title("Final-epoch validation mAP"); ax.set_ylabel("mAP")
ax.set_ylim(0, 1.08); ax.legend(fontsize=8); ax.grid(axis="x", visible=False)

show_fig(fig, "08_run_comparison")

comp = pd.DataFrame(RUN_REGISTRY).T[["weights", "freeze", "lr0", "epochs", "train_minutes"]]
comp["val_mAP50"]    = [f[1] for f in finals]
comp["val_mAP50_95"] = [f[2] for f in finals]
display(comp.round(4))

In [ ]:
diag = []
for t in TAGS:
    df = HIST[t]
    ep = rcol(df, "epoch")
    vbox = rcol(df, "val/box_loss"); tbox = rcol(df, "train/box_loss")
    vcls = rcol(df, "val/cls_loss"); tcls = rcol(df, "train/cls_loss")
    m95  = rcol(df, "metrics/mAP50-95(B)")

    val_total   = (vbox + vcls) if (vbox is not None and vcls is not None) else None
    train_total = (tbox + tcls) if (tbox is not None and tcls is not None) else None

    still_improving = bool(val_total[-1] < val_total[max(0, len(val_total) - 2)]) \
                      if val_total is not None and len(val_total) > 1 else None
    peak_ep   = int(np.argmax(m95)) if m95 is not None else -1
    peaked_early = bool(m95 is not None and peak_ep < len(m95) - 1)
    gap = float(val_total[-1] - train_total[-1]) if (val_total is not None
                                                     and train_total is not None) else np.nan

    diag.append(dict(run=t,
                     final_train_loss=round(float(train_total[-1]), 4) if train_total is not None else np.nan,
                     final_val_loss=round(float(val_total[-1]), 4) if val_total is not None else np.nan,
                     gap_val_minus_train=round(gap, 4),
                     val_loss_still_falling=still_improving,
                     best_epoch=peak_ep + 1, last_epoch=len(df),
                     mAP_peaked_early=peaked_early))
diag = pd.DataFrame(diag)
display(diag)

fig, ax = plt.subplots(figsize=(8.5, 4.2))
x = np.arange(len(diag)); width = 0.38
b1 = ax.bar(x - width / 2, diag.final_train_loss, width, label="train (augmented)",
            color=CAT[0], edgecolor=SURFACE, linewidth=1.5)
b2 = ax.bar(x + width / 2, diag.final_val_loss, width, label="validation (clean)",
            color=CAT[1], edgecolor=SURFACE, linewidth=1.5)
bar_labels(ax, b1, fmt="%.2f"); bar_labels(ax, b2, fmt="%.2f")
ax.set_xticks(x); ax.set_xticklabels([t.split("_")[0] for t in diag.run])
ax.set_ylabel("box + cls loss, final epoch")
ax.set_title("Final losses  (remember: train batches are mosaic-augmented, so they are harder)")
ax.legend(); ax.grid(axis="x", visible=False)
show_fig(fig, "09_overfitting_diagnostic")

for _, r in diag.iterrows():
    if r.mAP_peaked_early and not r.val_loss_still_falling:
        verdict = ("OVERFITTING - validation mAP peaked at epoch %d of %d and validation loss "
                   "has stopped falling. More epochs will not help; more data, stronger "
                   "augmentation or more regularisation will." % (r.best_epoch, r.last_epoch))
    elif r.val_loss_still_falling and not r.mAP_peaked_early:
        verdict = ("UNDERFITTING (undertrained) - validation loss is still falling and mAP is "
                   "still climbing at the last epoch. The model has not converged; it simply "
                   "needs more epochs. Expected at EPOCHS=%d." % EPOCHS)
    else:
        verdict = ("MIXED / INCONCLUSIVE - too few epochs to separate the signals. Re-read this "
                   "cell after the 30-epoch run; that is when the diagnosis becomes meaningful.")
    print("\n[%s]\n%s" % (r.run, textwrap.fill(verdict, 92, initial_indent="  ",
                                               subsequent_indent="  ")))

In [ ]:
def show_image_files(paths, titles, name, ncols=2, height=6.0):
    paths = [p for p in paths if p and Path(p).exists()]
    if not paths:
        print("(no files found for '%s')" % name); return
    titles = titles[:len(paths)]
    nrows = int(math.ceil(len(paths) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(7.6 * ncols, height * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, p, t in zip(axes, paths, titles):
        ax.imshow(plt.imread(str(p))); ax.set_title(t, fontsize=10); ax.axis("off")
    for ax in axes[len(paths):]:
        ax.axis("off")
    show_fig(fig, name)

ref_tag_batches = "B_full_lr01" if "B_full_lr01" in RUN_REGISTRY else TAGS[0]
ref = Path(RUN_REGISTRY[ref_tag_batches]["save_dir"])
print("showing real batches from run:", ref_tag_batches)
show_image_files(
    sorted(ref.glob("train_batch*.jpg"))[:2] + sorted(ref.glob("val_batch0*.jpg"))[:2],
    ["training batch 0 (mosaic + HSV + flip applied)",
     "training batch 1",
     "validation batch - GROUND TRUTH",
     "validation batch - PREDICTIONS"],
    "10_real_batches", ncols=2, height=6.5)

In [ ]:
def evaluate(tag, split="test"):
    w = RUN_REGISTRY[tag]["best"]
    if not Path(w).exists():
        print("!! no weights for", tag); return None
    mdl = YOLO(w)
    m = mdl.val(data=str(DATA_YAML), split=split, imgsz=IMGSZ, batch=BATCH,
                device=DEVICE, workers=WORKERS,
                conf=0.001,
                iou=IOU_NMS, plots=True, verbose=False,
                project=str(RUNS_DIR), name="%s_%s_eval" % (tag, split), exist_ok=True)

    out = dict(run=tag, split=split,
               mAP50=float(m.box.map50), mAP50_95=float(m.box.map),
               precision=float(m.box.mp), recall=float(m.box.mr))
    try:
        out["f1"] = 2 * out["precision"] * out["recall"] / max(out["precision"] + out["recall"], 1e-9)
    except Exception:
        out["f1"] = np.nan
    per_class = {}
    try:
        for i, ci in enumerate(m.box.ap_class_index):
            per_class[CLASS_NAMES[int(ci)]] = dict(
                AP50=float(m.box.ap50[i]), AP50_95=float(m.box.ap[i]),
                precision=float(m.box.p[i]), recall=float(m.box.r[i]))
    except Exception as e:
        print("  (per-class extraction failed: %s)" % e)
    out["_per_class"] = per_class
    out["_speed"] = dict(m.speed)
    out["_eval_dir"] = str(getattr(m, "save_dir", RUNS_DIR / ("%s_%s_eval" % (tag, split))))
    del mdl; torch.cuda.empty_cache()
    return out

sel = []
for t in TAGS:
    df = HIST[t]
    v50, v95 = rcol(df, "metrics/mAP50(B)"), rcol(df, "metrics/mAP50-95(B)")
    fitness = 0.1 * np.asarray(v50) + 0.9 * np.asarray(v95)
    sel.append(dict(run=t, best_val_fitness=float(fitness.max()),
                    best_epoch=int(np.argmax(fitness)) + 1,
                    val_mAP50=float(v50[-1]), val_mAP50_95=float(v95[-1])))
sel = pd.DataFrame(sel).sort_values("best_val_fitness", ascending=False).reset_index(drop=True)
BEST_TAG = sel.iloc[0]["run"]
print("Model selection (validation only):")
display(sel.round(4))
print(">>> WINNER: %s   (selected on validation fitness, before ever looking at test)\n" % BEST_TAG)

EVALS = {}
for t in TAGS:
    print("evaluating %s on the test split ..." % t)
    r = evaluate(t, TEST_SPLIT)
    if r:
        EVALS[t] = r

test_tbl = pd.DataFrame([{k: v for k, v in e.items() if not k.startswith("_")}
                         for e in EVALS.values()]).set_index("run")
test_tbl["is_selected"] = test_tbl.index == BEST_TAG
display(test_tbl.round(4))

In [ ]:
assert BEST_TAG in EVALS, ("The selected run %r has no test evaluation - check the "
                           "traceback from the previous cell." % BEST_TAG)
best_eval = EVALS[BEST_TAG]
assert best_eval["_per_class"], ("Per-class metrics could not be extracted from Ultralytics. "
                                 "The overall mAP in the table above is still valid; this cell "
                                 "needs m.box.ap_class_index, which your version may name "
                                 "differently.")
pc = pd.DataFrame(best_eval["_per_class"]).T
n_test = ann_df[ann_df.split == "test"].groupby("cls", observed=True).size()
n_train = ann_df[ann_df.split == "train"].groupby("cls", observed=True).size()
pc["test_boxes"]  = [int(n_test.get(c, 0)) for c in pc.index]
pc["train_boxes"] = [int(n_train.get(c, 0)) for c in pc.index]
pc.index = [dname(c) for c in pc.index]
pc = pc.sort_values("AP50_95", ascending=False)
display(pc.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))

ax = axes[0]
x = np.arange(len(pc)); width = 0.38
b1 = ax.bar(x - width / 2, pc.AP50,    width, label="AP@0.5",
            color=CAT[0], edgecolor=SURFACE, linewidth=1.5)
b2 = ax.bar(x + width / 2, pc.AP50_95, width, label="AP@0.5:0.95",
            color=CAT[1], edgecolor=SURFACE, linewidth=1.5)
bar_labels(ax, b1, fmt="%.3f"); bar_labels(ax, b2, fmt="%.3f")
ax.set_xticks(x); ax.set_xticklabels(pc.index, rotation=12, ha="right")
ax.set_ylabel("AP"); ax.set_ylim(0, 1.1); ax.legend()
ax.set_title("Per-class AP on the test split  (%s)" % BEST_TAG)
ax.grid(axis="x", visible=False)

ax = axes[1]
MARKERS = ["o", "s", "^", "D", "v", "P"]
for i, (nm, row) in enumerate(pc.iterrows()):
    ax.scatter(row.train_boxes, row.AP50_95, s=170,
               color=DISPLAY_COLOR.get(nm, CAT[i % len(CAT)]),
               marker=MARKERS[i % len(MARKERS)], edgecolors=SURFACE, linewidth=2, zorder=3)
    ax.annotate(nm, (row.train_boxes, row.AP50_95), xytext=(9, 4),
                textcoords="offset points", fontsize=9, color=INK2)
ax.set_xlabel("training boxes for this class"); ax.set_ylabel("test AP@0.5:0.95")
ax.set_title("Does class frequency predict accuracy?")
ax.set_ylim(0, 1.05)
ax.margins(x=0.22)

show_fig(fig, "11_per_class_ap")

worst = pc.AP50_95.idxmin(); best_c = pc.AP50_95.idxmax()
print("Best class : %-24s AP@0.5:0.95 = %.3f  (%d training boxes)"
      % (best_c, pc.loc[best_c, "AP50_95"], pc.loc[best_c, "train_boxes"]))
print("Worst class: %-24s AP@0.5:0.95 = %.3f  (%d training boxes)"
      % (worst, pc.loc[worst, "AP50_95"], pc.loc[worst, "train_boxes"]))

In [ ]:
ed = Path(best_eval["_eval_dir"])
print("evaluation artefacts in:", ed)
print(sorted(p.name for p in ed.glob("*.png")))

show_image_files(
    [next(iter(sorted(ed.glob("confusion_matrix_normalized.png"))), None),
     next(iter(sorted(ed.glob("confusion_matrix.png"))), None)],
    ["Confusion matrix, normalised by true class (column-wise)",
     "Confusion matrix, raw counts"],
    "12_confusion_matrix", ncols=2, height=6.2)

show_image_files(
    [next(iter(sorted(ed.glob("*PR_curve.png"))), None),
     next(iter(sorted(ed.glob("*F1_curve.png"))), None),
     next(iter(sorted(ed.glob("*P_curve.png"))), None),
     next(iter(sorted(ed.glob("*R_curve.png"))), None)],
    ["Precision-Recall curve (area under it = AP)",
     "F1 vs confidence - the peak tells you the best deployment threshold",
     "Precision vs confidence", "Recall vs confidence"],
    "13_pr_f1_curves", ncols=2, height=5.0)

In [ ]:
BENCH_N = min(60, len(test_images))
bench_imgs = [imread(p) for p in test_images[:BENCH_N]]
print("benchmarking on %d test images (pre-loaded into RAM so disk I/O is not timed)\n" % BENCH_N)

def bench(weights, device=DEVICE, half=False, warmup=10):
    mdl = YOLO(weights)
    n_par = sum(p.numel() for p in mdl.model.parameters()) / 1e6

    for _ in range(warmup):
        mdl.predict(bench_imgs[0], imgsz=IMGSZ, device=device, half=half, verbose=False)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    acc = {"preprocess": 0.0, "inference": 0.0, "postprocess": 0.0}
    t0 = time.perf_counter()
    for a in bench_imgs:
        mdl.predict(a, imgsz=IMGSZ, device=device, half=half, verbose=False)
        sp = getattr(mdl.predictor, "speed", None) or {}
        for k in acc:
            acc[k] += float(sp.get(k, 0.0))
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    n = len(bench_imgs)
    del mdl; torch.cuda.empty_cache()
    return dict(ms_per_image=1000 * dt / n,
                fps=n / dt,
                net_inference_ms=acc["inference"] / n,
                preprocess_ms=acc["preprocess"] / n,
                postprocess_ms=acc["postprocess"] / n,
                params_M=round(n_par, 2))

speed_rows = []
for t in TAGS:
    r = bench(RUN_REGISTRY[t]["best"], device=DEVICE, half=False)
    r.update(run=t, precision="FP32")
    speed_rows.append(r)

if torch.cuda.is_available():
    r = bench(RUN_REGISTRY[BEST_TAG]["best"], device=DEVICE, half=True)
    r.update(run=BEST_TAG + " (FP16)", precision="FP16")
    speed_rows.append(r)

cpu_imgs_backup = bench_imgs
try:
    bench_imgs = bench_imgs[:12]
    r = bench(RUN_REGISTRY[BEST_TAG]["best"], device="cpu", half=False, warmup=2)
    r.update(run=BEST_TAG + " (CPU)", precision="FP32-CPU")
    speed_rows.append(r)
except Exception as e:
    print("CPU benchmark skipped:", e)
finally:
    bench_imgs = cpu_imgs_backup

speed_tbl = pd.DataFrame(speed_rows).set_index("run")[
    ["precision", "params_M", "ms_per_image", "fps",
     "preprocess_ms", "net_inference_ms", "postprocess_ms"]]
display(speed_tbl.round(2))

pts = []
for t in TAGS:
    if t in EVALS:
        row = speed_tbl.loc[t]
        pts.append((t, float(row.ms_per_image), EVALS[t]["mAP50_95"], EVALS[t]["mAP50"]))

fig, ax = plt.subplots(figsize=(8.6, 5.4))
for i, (t, ms, m95, m50) in enumerate(pts):
    ax.scatter(ms, m95, s=230, color=CAT[i % len(CAT)], marker=MARKERS[i % len(MARKERS)],
               edgecolors=SURFACE, linewidth=2.2, zorder=3)
    ax.annotate("%s\n%.1f ms | mAP %.3f" % (t.split("_")[0], ms, m95),
                (ms, m95), xytext=(11, -4), textcoords="offset points",
                fontsize=9, color=INK2)
ax.set_xlabel("end-to-end latency per image (ms, lower is better)")
ax.set_ylabel("test mAP@0.5:0.95 (higher is better)")
ax.set_title("Speed / accuracy trade-off - top-left is the ideal corner")
ax.margins(x=0.3, y=0.25)
show_fig(fig, "14_speed_vs_accuracy")

In [ ]:
best_model = YOLO(RUN_REGISTRY[BEST_TAG]["best"])
print("Loaded selected model:", RUN_REGISTRY[BEST_TAG]["best"])

def load_gt_boxes(split_key, image_name, W, H):
    lp = splits[split_key] / "labels" / (Path(image_name).stem + ".txt")
    boxes, ids = [], []
    if lp.exists():
        for ln in lp.read_text().strip().splitlines():
            p = ln.split()
            if len(p) < 5:
                continue
            try:
                c = int(float(p[0])); vals = [float(v) for v in p[1:]]
            except ValueError:
                continue
            if len(p) == 5:
                cx, cy, w, h = vals
            else:
                xs, ys = vals[0::2], vals[1::2]
                cx, cy = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2
                w,  h  = max(xs) - min(xs),      max(ys) - min(ys)
            if w <= 0 or h <= 0 or not (0 <= c < NC):
                continue
            boxes.append(yolo_to_xyxy(cx, cy, w, h, W, H)); ids.append(c)
    return np.asarray(boxes, np.float32).reshape(-1, 4), np.asarray(ids, int)

rng = random.Random(SEED + 1)
demo = rng.sample(test_images, min(4, len(test_images)))

fig, axes = plt.subplots(len(demo), 2, figsize=(13, 6.4 * len(demo)))
axes = np.atleast_2d(axes)
for r_i, ip in enumerate(demo):
    img = imread(ip); H, W = img.shape[:2]
    gb, gi = load_gt_boxes(TEST_SPLIT, ip.name, W, H)
    vis_gt = draw_boxes(img, gb, [dname(CLASS_NAMES[c]) for c in gi],
                        [hex2bgr(CLASS_COLOR[CLASS_NAMES[c]]) for c in gi])

    res = best_model.predict(img, imgsz=IMGSZ, conf=CONF_DEPLOY, iou=IOU_NMS,
                             device=DEVICE, verbose=False)[0]
    pb = res.boxes.xyxy.cpu().numpy()
    pc_ = res.boxes.cls.cpu().numpy().astype(int)
    pf = res.boxes.conf.cpu().numpy()
    vis_pr = draw_boxes(img, pb,
                        ["%s %.2f" % (dname(CLASS_NAMES[c]), s) for c, s in zip(pc_, pf)],
                        [hex2bgr(CLASS_COLOR[CLASS_NAMES[c]]) for c in pc_])

    axes[r_i][0].imshow(cv2.cvtColor(vis_gt, cv2.COLOR_BGR2RGB))
    axes[r_i][0].set_title("GROUND TRUTH  -  %s  (%d objects)" % (ip.name[:26], len(gb)), fontsize=10)
    axes[r_i][1].imshow(cv2.cvtColor(vis_pr, cv2.COLOR_BGR2RGB))
    axes[r_i][1].set_title("PREDICTION  -  %d detections @ conf>=%.2f" % (len(pb), CONF_DEPLOY),
                           fontsize=10)
    axes[r_i][0].axis("off"); axes[r_i][1].axis("off")
handles = [Patch(facecolor=CLASS_COLOR[c], label=dname(c)) for c in cls_order]
fig.legend(handles=handles, loc="lower center", ncol=len(cls_order), bbox_to_anchor=(0.5, -0.005))
show_fig(fig, "15_predictions_vs_gt")

In [ ]:
LOW_CONF = 0.01

RAW_PREDS = {}
t0 = time.perf_counter()
stream = best_model.predict(source=[str(p) for p in test_images], imgsz=IMGSZ,
                            conf=LOW_CONF, iou=IOU_NMS, device=DEVICE,
                            stream=True, verbose=False)
for k, res in enumerate(stream):
    nm = Path(res.path).name
    b = res.boxes
    RAW_PREDS[nm] = dict(
        boxes=b.xyxy.cpu().numpy().astype(np.float32) if b is not None else np.zeros((0, 4), np.float32),
        cls=b.cls.cpu().numpy().astype(int) if b is not None else np.zeros(0, int),
        conf=b.conf.cpu().numpy().astype(np.float32) if b is not None else np.zeros(0, np.float32),
        H=res.orig_shape[0], W=res.orig_shape[1])
    if (k + 1) % 60 == 0:
        print("  %d/%d images" % (k + 1, len(test_images)))
print("inference over %d test images in %.1f s" % (len(RAW_PREDS), time.perf_counter() - t0))

GT = {}
for ip in test_images:
    meta = RAW_PREDS.get(ip.name)
    if meta is None:
        continue
    gb, gi = load_gt_boxes(TEST_SPLIT, ip.name, meta["W"], meta["H"])
    GT[ip.name] = dict(boxes=gb, cls=gi)
print("ground truth loaded for %d images (%d boxes total)"
      % (len(GT), sum(len(v["cls"]) for v in GT.values())))

In [ ]:
def iou_matrix(a, b):
    a = np.asarray(a, np.float32).reshape(-1, 4)
    b = np.asarray(b, np.float32).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    aa = np.clip(a[:, 2] - a[:, 0], 0, None) * np.clip(a[:, 3] - a[:, 1], 0, None)
    bb = np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)
    return inter / (aa[:, None] + bb[None, :] - inter + 1e-9)

def match_image(pb, pc_, pf, gb, gc):
    gc = np.asarray(gc, int)
    M = iou_matrix(pb, gb)
    gt_used   = np.zeros(len(gb), bool)
    pred_done = np.zeros(len(pb), bool)
    ev = []
    order = np.argsort(-np.asarray(pf, dtype=np.float64)) if len(pb) else np.zeros(0, int)

    for pi in order:
        row = M[pi] if M.size else np.zeros(0, np.float32)
        if not len(row):
            continue
        same = np.where((~gt_used) & (gc == pc_[pi]), row, -1.0)
        si = int(np.argmax(same))
        if float(same[si]) >= IOU_MATCH:
            gt_used[si] = True
            pred_done[pi] = True
            ev.append(("TP", pi, si, float(same[si])))

    for pi in order:
        if pred_done[pi]:
            continue
        row = M[pi] if M.size else np.zeros(0, np.float32)
        if len(row):
            avail = ~gt_used
            same  = np.where(avail & (gc == pc_[pi]), row, -1.0)
            other = np.where(avail & (gc != pc_[pi]), row, -1.0)
            si = int(np.argmax(same));  s_iou = float(same[si])
            oi = int(np.argmax(other)); o_iou = float(other[oi])
        else:
            si = oi = -1; s_iou = o_iou = -1.0

        if o_iou >= IOU_MATCH:
            gt_used[oi] = True; ev.append(("CLASS", pi, oi, o_iou))
        elif s_iou >= IOU_LOC:
            gt_used[si] = True; ev.append(("LOC", pi, si, s_iou))
        elif o_iou >= IOU_LOC:
            gt_used[oi] = True; ev.append(("CLASS_LOC", pi, oi, o_iou))
        else:
            ev.append(("GHOST", pi, -1, max(s_iou, o_iou, 0.0)))

    for gi in np.flatnonzero(~gt_used):
        ev.append(("MISS", -1, gi, 0.0))
    return ev

def filter_preds(rec, conf_thr):
    m = rec["conf"] >= conf_thr
    return rec["boxes"][m], rec["cls"][m], rec["conf"][m]

def run_analysis(conf_thr):
    rows = []
    for nm, rec in RAW_PREDS.items():
        g = GT.get(nm, dict(boxes=np.zeros((0, 4), np.float32), cls=np.zeros(0, int)))
        pb, pc_, pf = filter_preds(rec, conf_thr)
        for kind, pi, gi, iou in match_image(pb, pc_, pf, g["boxes"], g["cls"]):
            rows.append(dict(
                image=nm, kind=kind, iou=round(float(iou), 4),
                pred_cls=CLASS_NAMES[int(pc_[pi])] if pi >= 0 else None,
                pred_conf=round(float(pf[pi]), 4) if pi >= 0 else np.nan,
                gt_cls=CLASS_NAMES[int(g["cls"][gi])] if gi >= 0 else None,
                pred_idx=pi, gt_idx=gi))
    return pd.DataFrame(rows)

EV = run_analysis(CONF_DEPLOY)
ERROR_KINDS = ["CLASS", "CLASS_LOC", "LOC", "GHOST", "MISS"]

OUTCOME_COUNTS = EV.kind.value_counts()
n_gt   = sum(len(v["cls"]) for v in GT.values())
n_pred = int((EV.pred_idx >= 0).sum())
n_tp   = int(OUTCOME_COUNTS.get("TP", 0))

_gt_events = int(EV.gt_idx.ge(0).sum())
assert _gt_events == n_gt, ("matcher accounted for %d ground-truth boxes but the test set "
                            "has %d - the outcome bookkeeping is broken" % (_gt_events, n_gt))

print("TEST SET, at deployment threshold conf >= %.2f" % CONF_DEPLOY)
print("-" * 62)
print("  ground-truth objects        : %d" % n_gt)
print("  detections produced         : %d" % n_pred)
print("  correct (TP)                : %d" % n_tp)
for k in ERROR_KINDS:
    print("  %-27s : %d" % (k, int(OUTCOME_COUNTS.get(k, 0))))
print("-" * 62)
prec = n_tp / max(n_pred, 1)
rec  = n_tp / max(n_gt, 1)
print("  precision  TP/(all detections) = %.4f" % prec)
print("  recall     TP/(all GT objects) = %.4f" % rec)
print("  F1                             = %.4f" % (2 * prec * rec / max(prec + rec, 1e-9)))

In [ ]:
KIND_COLOR = {"TP": STATUS["good"], "LOC": STATUS["warning"], "CLASS": STATUS["serious"],
              "CLASS_LOC": STATUS["serious"], "GHOST": STATUS["critical"], "MISS": STATUS["critical"]}
KIND_LABEL = {"TP": "correct", "LOC": "sloppy box", "CLASS": "wrong class",
              "CLASS_LOC": "wrong class + box", "GHOST": "hallucinated", "MISS": "missed"}

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))

ax = axes[0]
order = ["TP"] + ERROR_KINDS
vals = [int(OUTCOME_COUNTS.get(k, 0)) for k in order]
bars = ax.bar([KIND_LABEL[k] for k in order], vals,
              color=[KIND_COLOR[k] for k in order], edgecolor=SURFACE, linewidth=1.5, width=0.62)
bar_labels(ax, bars)
ax.set_ylabel("count"); ax.set_title("Outcome breakdown at conf >= %.2f" % CONF_DEPLOY)
ax.tick_params(axis="x", labelrotation=16)
ax.grid(axis="x", visible=False)

ax = axes[1]
gt_side = EV[EV.gt_idx >= 0]
if len(gt_side):
    pivot = (gt_side.groupby(["gt_cls", "kind"], observed=True).size().unstack(fill_value=0)
             .reindex(columns=["TP", "LOC", "CLASS", "CLASS_LOC", "MISS"], fill_value=0)
             .reindex(index=[c for c in cls_order if c in set(gt_side.gt_cls)], fill_value=0))
else:
    pivot = pd.DataFrame(0, index=cls_order,
                         columns=["TP", "LOC", "CLASS", "CLASS_LOC", "MISS"])
bottom = np.zeros(len(pivot))
for k in ["TP", "LOC", "CLASS", "CLASS_LOC", "MISS"]:
    v = pivot[k].values
    ax.bar([dname(c) for c in pivot.index], v, bottom=bottom, label=KIND_LABEL[k],
           color=KIND_COLOR[k], edgecolor=SURFACE, linewidth=1.5, width=0.6)
    bottom += v
ax.set_ylabel("ground-truth objects"); ax.set_title("What happened to each true object, by class")
ax.legend(fontsize=8, ncol=2); ax.grid(axis="x", visible=False)
ax.tick_params(axis="x", labelrotation=12)
show_fig(fig, "16_error_breakdown")

recall_tbl = pivot.copy()
recall_tbl["total"] = recall_tbl.sum(axis=1)
recall_tbl["recall"] = (recall_tbl["TP"] / recall_tbl["total"]).round(3)
recall_tbl["miss_rate"] = (recall_tbl["MISS"] / recall_tbl["total"]).round(3)
recall_tbl.index = [dname(c) for c in recall_tbl.index]
display(recall_tbl)

conf_pairs = EV[EV.kind.isin(["CLASS", "CLASS_LOC"])]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

ax = axes[0]
if len(conf_pairs):
    cm = (conf_pairs.groupby(["gt_cls", "pred_cls"], observed=True).size()
          .unstack(fill_value=0).reindex(index=cls_order, columns=cls_order, fill_value=0))
    im = ax.imshow(cm.values, cmap=SEQ)
    ax.set_xticks(range(NC)); ax.set_xticklabels([dname(c) for c in cls_order], rotation=20, ha="right")
    ax.set_yticks(range(NC)); ax.set_yticklabels([dname(c) for c in cls_order])
    mx = max(cm.values.max(), 1)
    for i in range(NC):
        for j in range(NC):
            ax.text(j, i, str(cm.values[i, j]), ha="center", va="center", fontsize=10,
                    color="#ffffff" if cm.values[i, j] > 0.55 * mx else INK)
    ax.set_xlabel("predicted as"); ax.set_ylabel("true class")
    plt.colorbar(im, ax=ax, fraction=0.046)
else:
    ax.text(0.5, 0.5, "no class-confusion errors", ha="center", va="center", color=MUTED)
ax.set_title("Class confusions only"); ax.grid(False)

ax = axes[1]
tp_iou = EV.loc[EV.kind == "TP", "iou"].values
loc_iou = EV.loc[EV.kind.isin(["LOC", "CLASS_LOC"]), "iou"].values
if len(tp_iou):
    ax.hist(tp_iou, bins=np.linspace(0.5, 1.0, 21), color=STATUS["good"],
            edgecolor=SURFACE, linewidth=1.2, label="true positives")
if len(loc_iou):
    ax.hist(loc_iou, bins=np.linspace(0.0, 0.5, 11), color=STATUS["warning"],
            edgecolor=SURFACE, linewidth=1.2, label="localisation errors")
ax.axvline(IOU_MATCH, color=INK2, ls="--", lw=1.6)
ax.annotate("IoU=0.50 cut", (IOU_MATCH, ax.get_ylim()[1] * 0.92), xytext=(5, 0),
            textcoords="offset points", fontsize=8, color=INK2)
ax.set_xlabel("IoU with matched ground truth"); ax.set_ylabel("detections")
ax.set_title("Localisation quality"); ax.legend(fontsize=8); ax.grid(axis="x", visible=False)

ax = axes[2]
tp_c = EV.loc[EV.kind == "TP", "pred_conf"].dropna().values
fp_c = EV.loc[EV.kind.isin(["GHOST", "CLASS", "CLASS_LOC", "LOC"]), "pred_conf"].dropna().values
bins = np.linspace(CONF_DEPLOY, 1.0, 16)
if len(tp_c):
    ax.hist(tp_c, bins=bins, histtype="step", lw=2.2, color=STATUS["good"], label="correct")
if len(fp_c):
    ax.hist(fp_c, bins=bins, histtype="step", lw=2.2, color=STATUS["critical"],
            label="wrong or sloppy")
ax.set_xlabel("model confidence"); ax.set_ylabel("detections")
ax.set_title("Is the model confident when it is wrong?")
ax.legend(fontsize=8); ax.grid(axis="x", visible=False)

show_fig(fig, "17_error_analysis_detail")

if len(tp_c) and len(fp_c):
    print("mean confidence when CORRECT : %.3f" % tp_c.mean())
    print("mean confidence when WRONG   : %.3f" % fp_c.mean())

In [ ]:
N_ERROR_CASES = 6

SEVERITY = {"MISS": 3, "CLASS": 3, "CLASS_LOC": 3, "GHOST": 2, "LOC": 1}
err = EV[EV.kind.isin(ERROR_KINDS)].copy()
err["severity"] = err.kind.map(SEVERITY)
worst = (err.groupby("image").agg(n_err=("kind", "size"), score=("severity", "sum"))
         .sort_values(["score", "n_err"], ascending=False))

DIAGNOSIS = {
    "MISS": ("The model produced NO detection for this object at conf>=%.2f." % CONF_DEPLOY,
             "Check its size in the image. Small, occluded or heavily cropped objects are the "
             "usual cause; a rare class is the second. Fixes: train longer / higher imgsz / "
             "oversample this class / lower the deployment threshold (costs precision)."),
    "CLASS": ("The object WAS found (good box) but given the WRONG LABEL.",
              "This is a semantics problem, not a vision problem. For this dataset 'person' and "
              "'people_wheelchair' are nested categories - a seated person is still a person - so "
              "the decision boundary is a labelling convention. Fixes: clarify the annotation "
              "rule, merge the classes, or add a hierarchical/multi-label head."),
    "CLASS_LOC": ("Wrong label AND a poor box - the region is genuinely confusing.",
                  "Often occlusion or two overlapping people. Inspect the crop before blaming "
                  "the model; the ground truth may itself be debatable."),
    "LOC": ("Right class, but the box is too loose or too tight (IoU below 0.50).",
            "A regression problem. Common on partially occluded objects and on objects near the "
            "image border. Fixes: higher input resolution, less aggressive rotation/scale "
            "augmentation, longer training."),
    "GHOST": ("The model invented an object where there is nothing labelled.",
              "Either a genuine hallucination on background clutter, or - check this first - a "
              "MISSING ANNOTATION in the ground truth. Unlabelled true objects are common in "
              "crowd-sourced datasets and unfairly punish a correct model."),
}

cases = list(worst.head(N_ERROR_CASES).index) if len(worst) else []
if not cases:
    print("No errors at all on the test set at conf >= %.2f." % CONF_DEPLOY)

print("=" * 94)
print(" %d WORST TEST IMAGES  (model: %s, conf >= %.2f)" % (len(cases), BEST_TAG, CONF_DEPLOY))
print("=" * 94)

for n, nm in enumerate(cases, 1):
    sub = EV[EV.image == nm]
    rec = RAW_PREDS[nm]
    print("\n" + "-" * 94)
    print("ERROR CASE %d/%d   image: %s   (%dx%d px)"
          % (n, len(cases), nm, rec["W"], rec["H"]))
    print("  ground-truth objects: %d   |   detections at conf>=%.2f: %d   |   errors: %d"
          % (len(GT[nm]["cls"]), CONF_DEPLOY,
             int((sub.pred_idx >= 0).sum()), int(sub.kind.isin(ERROR_KINDS).sum())))
    print("-" * 94)
    for _, e in sub.iterrows():
        if e.kind == "TP":
            print("   OK      %-22s conf=%.2f  IoU=%.2f" % (dname(e.pred_cls), e.pred_conf, e.iou))
    for _, e in sub[sub.kind.isin(ERROR_KINDS)].iterrows():
        head, why = DIAGNOSIS[e.kind]
        if e.kind == "MISS":
            what = "true '%s' was NOT DETECTED" % dname(e.gt_cls)
        elif e.kind == "GHOST":
            what = "predicted '%s' (conf=%.2f) matches no ground truth" % (dname(e.pred_cls), e.pred_conf)
        elif e.kind == "LOC":
            what = "'%s' (conf=%.2f) located at IoU=%.2f - below the 0.50 bar" % (
                dname(e.pred_cls), e.pred_conf, e.iou)
        else:
            what = "true '%s' predicted as '%s' (conf=%.2f, IoU=%.2f)" % (
                dname(e.gt_cls), dname(e.pred_cls), e.pred_conf, e.iou)
        print("\n   >> %-10s %s" % (e.kind, what))
        print("      %s" % head)
        print(textwrap.fill(why, 88, initial_indent="      why: ", subsequent_indent="           "))

fig, axes = plt.subplots(max(len(cases), 1), 2, figsize=(13, 6.4 * max(len(cases), 1)))
axes = np.atleast_2d(axes)
for r_i, nm in enumerate(cases):
    img = imread(splits[TEST_SPLIT] / "images" / nm)
    g = GT[nm]
    pb, pc_, pf = filter_preds(RAW_PREDS[nm], CONF_DEPLOY)
    kinds = EV[(EV.image == nm) & (EV.pred_idx >= 0)].set_index("pred_idx")["kind"].to_dict()

    vis_gt = draw_boxes(img, g["boxes"], [dname(CLASS_NAMES[c]) for c in g["cls"]],
                        [hex2bgr(STATUS["good"])] * len(g["cls"]))
    vis_pr = draw_boxes(img, pb,
                        ["%s %.2f [%s]" % (dname(CLASS_NAMES[c]), s, kinds.get(i, "?"))
                         for i, (c, s) in enumerate(zip(pc_, pf))],
                        [hex2bgr(KIND_COLOR.get(kinds.get(i, "GHOST"), STATUS["critical"]))
                         for i in range(len(pb))])
    n_err = int(EV[(EV.image == nm) & EV.kind.isin(ERROR_KINDS)].shape[0])
    axes[r_i][0].imshow(cv2.cvtColor(vis_gt, cv2.COLOR_BGR2RGB))
    axes[r_i][0].set_title("CASE %d  GROUND TRUTH  -  %s" % (r_i + 1, nm[:30]), fontsize=10)
    axes[r_i][1].imshow(cv2.cvtColor(vis_pr, cv2.COLOR_BGR2RGB))
    axes[r_i][1].set_title("PREDICTION  -  %d error(s); box colour = error type" % n_err, fontsize=10)
    axes[r_i][0].axis("off"); axes[r_i][1].axis("off")
for ax in axes.ravel()[2 * len(cases):]:
    ax.axis("off")
handles = [Patch(facecolor=KIND_COLOR[k], label=KIND_LABEL[k])
           for k in ["TP", "LOC", "CLASS", "GHOST"]]
fig.legend(handles=handles, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.004))
show_fig(fig, "18_failure_cases")

In [ ]:
sweep = []
for thr in np.round(np.arange(0.05, 0.91, 0.05), 2):
    tp = fp = 0
    for nm, rec in RAW_PREDS.items():
        g = GT.get(nm, dict(boxes=np.zeros((0, 4), np.float32), cls=np.zeros(0, int)))
        pb, pc_, pf = filter_preds(rec, thr)
        for kind, pi, gi, iou in match_image(pb, pc_, pf, g["boxes"], g["cls"]):
            if kind == "TP":
                tp += 1
            elif pi >= 0:
                fp += 1
    p = tp / max(tp + fp, 1)
    r = tp / max(n_gt, 1)
    sweep.append(dict(conf=thr, precision=p, recall=r,
                      f1=2 * p * r / max(p + r, 1e-9), tp=tp, fp=fp,
                      fn=n_gt - tp))
sweep = pd.DataFrame(sweep)
best_row = sweep.loc[sweep.f1.idxmax()]

fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.6))
ax = axes[0]
for i, (c, lab) in enumerate([("precision", "precision"), ("recall", "recall"), ("f1", "F1")]):
    ax.plot(sweep.conf, sweep[c], color=CAT[i], marker="o", ms=3.5, label=lab)
ax.axvline(best_row.conf, color=INK2, ls="--", lw=1.5)
ax.annotate("best F1 = %.3f\nat conf = %.2f" % (best_row.f1, best_row.conf),
            (best_row.conf, 0.08), xytext=(8, 0), textcoords="offset points",
            fontsize=9, color=INK2)
ax.axvline(CONF_DEPLOY, color=MUTED, ls=":", lw=1.5)
ax.annotate("default 0.25", (CONF_DEPLOY, 0.98), xytext=(-6, 0), textcoords="offset points",
            fontsize=8, color=MUTED, ha="right")
ax.set_xlabel("confidence threshold"); ax.set_ylabel("score")
ax.set_ylim(0, 1.05); ax.set_title("Operating point selection"); ax.legend(fontsize=8)

ax = axes[1]
for i, (c, lab) in enumerate([("tp", "true positives"), ("fp", "false positives"),
                              ("fn", "missed objects")]):
    ax.plot(sweep.conf, sweep[c], color=[STATUS["good"], STATUS["critical"], STATUS["warning"]][i],
            marker="o", ms=3.5, label=lab)
ax.set_xlabel("confidence threshold"); ax.set_ylabel("count")
ax.set_title("Raising the threshold trades misses for false alarms"); ax.legend(fontsize=8)
show_fig(fig, "19_threshold_sweep")

display(sweep.round(3))

In [ ]:
try:
    core = best_model.model
    seq_b = core.model
    probe_idx = [i for i in (0, 2, 4, 6) if i < len(seq_b)]
    acts, hooks = {}, []

    def make_hook(i):
        def fn(mod, inp, out):
            if isinstance(out, torch.Tensor):
                acts[i] = out.detach().float().cpu()
        return fn

    for i in probe_idx:
        hooks.append(seq_b[i].register_forward_hook(make_hook(i)))

    sample_path = str(test_images[0])
    _ = best_model.predict(sample_path, imgsz=IMGSZ, device=DEVICE, verbose=False)
    for h in hooks:
        h.remove()

    sample_img = cv2.cvtColor(imread(sample_path), cv2.COLOR_BGR2RGB)
    keys = [i for i in probe_idx if i in acts]
    ncols = 6
    fig, axes = plt.subplots(len(keys), ncols, figsize=(2.5 * ncols, 2.7 * len(keys)))
    axes = np.atleast_2d(axes)
    for r_i, i in enumerate(keys):
        a = acts[i][0]
        axes[r_i][0].imshow(a.mean(0).numpy(), cmap=SEQ)
        axes[r_i][0].set_ylabel("layer %d\n%s" % (i, type(seq_b[i]).__name__),
                                fontsize=8, color=INK2)
        axes[r_i][0].set_title("mean of %d channels" % a.shape[0], fontsize=8)
        for c in range(1, ncols):
            ch = min(c - 1, a.shape[0] - 1)
            axes[r_i][c].imshow(a[ch].numpy(), cmap=SEQ)
            axes[r_i][c].set_title("channel %d" % ch, fontsize=8)
        for c in range(ncols):
            axes[r_i][c].set_xticks([]); axes[r_i][c].set_yticks([]); axes[r_i][c].grid(False)
    fig.suptitle("Backbone activations  (input: %s)" % Path(sample_path).name,
                 x=0.09, ha="left", fontsize=12, fontweight="bold")
    show_fig(fig, "20_feature_maps")

    fig2, ax = plt.subplots(figsize=(5.2, 5.2))
    ax.imshow(sample_img); ax.axis("off"); ax.set_title("the input image", fontsize=10)
    show_fig(fig2, "20b_feature_map_input")

except Exception as e:
    print("Feature-map visualisation skipped:", type(e).__name__, e)

In [ ]:
best_ev = EVALS[BEST_TAG]
ref_tag = "B_full_lr01"

def g(tag, key, default=np.nan):
    return EVALS[tag][key] if tag in EVALS else default

lines = []
lines.append(" PROJECT SUMMARY  -  generated %s" % time.strftime("%Y-%m-%d %H:%M"))
lines.append("")
lines.append("  source        %s / %s  version %s" % (RF_WORKSPACE, RF_PROJECT, RF_VERSION))
lines.append("  images        train %d | val %d | test %d  (total %d)"
             % tuple([int(img_df[img_df.split == s].shape[0]) for s in SPLIT_ORDER]
                     + [int(len(img_df))]))
lines.append("  boxes         " + " | ".join(
    "%s %d" % (dname(c), int(v)) for c, v in
    ann_df.groupby("cls", observed=True).size().items()) + "   (total %d)" % len(ann_df))
lines.append("  imbalance     %.1f : 1 (most vs least frequent class in train)" % imb)
lines.append("  test split    %s" % ("shipped by the export - held out at project level, never carved by us"
                                     if TEST_SPLIT_IS_NATIVE else
                                     "CARVED BY US from valid (seed %d, %.0f%%) - val and test share a pool"
                                     % (SEED, 100 * TEST_FRACTION)))
lines.append("")
lines.append("TRAINING  (%d epochs, seed %d, %d px, batch %d, SGD, warmup %.1f)"
             % (EPOCHS, SEED, IMGSZ, BATCH, WARMUP_EPOCHS))
lines.append("  loss gains    box=%.1f  cls=%.1f  dfl=%.1f"
             % (BOX_GAIN, CLS_GAIN, DFL_GAIN))
for t in TAGS:
    m = RUN_REGISTRY[t]
    lines.append("  %-16s freeze=%-3d lr0=%-7g %-12s %5.1f min"
                 % (t, m["freeze"], m["lr0"], m["weights"], m["train_minutes"]))
lines.append("")
lines.append("SELECTED MODEL (chosen on VALIDATION fitness, before any test evaluation): %s" % BEST_TAG)
lines.append("")
lines.append("TEST-SET RESULTS")
lines.append("  %-18s %9s %9s %9s %9s" % ("run", "mAP50", "mAP50-95", "precision", "recall"))
for t in TAGS:
    if t in EVALS:
        e = EVALS[t]
        star = " <-- selected" if t == BEST_TAG else ""
        lines.append("  %-18s %9.4f %9.4f %9.4f %9.4f%s"
                     % (t, e["mAP50"], e["mAP50_95"], e["precision"], e["recall"], star))
lines.append("")
lines.append("PER-CLASS AP  (selected model)")
for nm, row in pc.iterrows():
    lines.append("  %-24s AP50=%.4f  AP50-95=%.4f  P=%.3f  R=%.3f  (%d train / %d test boxes)"
                 % (nm, row.AP50, row.AP50_95, row.precision, row.recall,
                    row.train_boxes, row.test_boxes))
lines.append("")
lines.append("ABLATIONS  (delta in test mAP@0.5:0.95)")
if "A_frozen_lr01" in EVALS and ref_tag in EVALS:
    d = g(ref_tag, "mAP50_95") - g("A_frozen_lr01", "mAP50_95")
    lines.append("  unfreezing the backbone (B - A) : %+.4f" % d)
if "C_full_lr001" in EVALS and ref_tag in EVALS:
    d = g("C_full_lr001", "mAP50_95") - g(ref_tag, "mAP50_95")
    lines.append("  lr0 0.01 -> 0.001    (C - B)    : %+.4f" % d)
if "D_yolo11s_lr01" in EVALS and ref_tag in EVALS:
    d = g("D_yolo11s_lr01", "mAP50_95") - g(ref_tag, "mAP50_95")
    ms_b = float(speed_tbl.loc[ref_tag, "ms_per_image"])
    ms_d = float(speed_tbl.loc["D_yolo11s_lr01", "ms_per_image"])
    lines.append("  yolo11n -> yolo11s   (D - B)    : %+.4f  for %+.1f ms/img (%.2fx slower)"
                 % (d, ms_d - ms_b, ms_d / max(ms_b, 1e-9)))
lines.append("")
lines.append("SPEED  (selected model, %s)" % ("T4 GPU" if torch.cuda.is_available() else "CPU"))
sr = speed_tbl.loc[BEST_TAG]
lines.append("  %.1f ms/image end-to-end  =  %.1f FPS" % (sr.ms_per_image, sr.fps))
lines.append("  of which network forward pass: %.1f ms" % sr.net_inference_ms)
lines.append("")
lines.append("ERROR PROFILE  (test, conf >= %.2f)" % CONF_DEPLOY)
lines.append("  precision %.4f | recall %.4f | F1 %.4f"
             % (prec, rec, 2 * prec * rec / max(prec + rec, 1e-9)))
for k in ERROR_KINDS:
    v = int(OUTCOME_COUNTS.get(k, 0))
    lines.append("  %-10s %4d  (%.1f%% of all errors)"
                 % (KIND_LABEL[k], v, 100 * v / max(int(EV.kind.isin(ERROR_KINDS).sum()), 1)))
lines.append("  best F1 threshold would be conf=%.2f (F1=%.3f) rather than the default %.2f"
             % (best_row.conf, best_row.f1, CONF_DEPLOY))
lines.append("")

SUMMARY = "\n".join(lines)
print(SUMMARY)
(PROJECT_DIR / "summary.txt").write_text(SUMMARY, encoding="utf-8")

In [ ]:
EXPORT_DIR = PROJECT_DIR / "submission"
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
(EXPORT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(EXPORT_DIR / "weights").mkdir(parents=True, exist_ok=True)
(EXPORT_DIR / "runs").mkdir(parents=True, exist_ok=True)

for p in sorted(FIG_DIR.glob("*.png")):
    shutil.copy2(p, EXPORT_DIR / "figures" / p.name)

bw = Path(RUN_REGISTRY[BEST_TAG]["best"])
if bw.exists():
    shutil.copy2(bw, EXPORT_DIR / "weights" / ("best_%s.pt" % BEST_TAG))

for t in TAGS:
    sd = Path(RUN_REGISTRY[t]["save_dir"])
    dst = EXPORT_DIR / "runs" / t
    dst.mkdir(parents=True, exist_ok=True)
    for pat in ("results.csv", "args.yaml", "results.png",
                "confusion_matrix_normalized.png", "labels.jpg"):
        for f in sd.glob(pat):
            shutil.copy2(f, dst / f.name)

test_tbl.round(4).to_csv(EXPORT_DIR / "test_metrics.csv")
pc.round(4).to_csv(EXPORT_DIR / "per_class_metrics.csv")
speed_tbl.round(3).to_csv(EXPORT_DIR / "speed_benchmark.csv")
sweep.round(4).to_csv(EXPORT_DIR / "threshold_sweep.csv")
EV.to_csv(EXPORT_DIR / "error_events.csv", index=False)
ann_df.to_csv(EXPORT_DIR / "annotations.csv", index=False)
shutil.copy2(PROJECT_DIR / "summary.txt", EXPORT_DIR / "summary.txt")

REQUIREMENTS = "\n".join([
    "# Python 3.10+   |   install with:  pip install -r requirements.txt",
    "ultralytics>=8.3.0",
    "roboflow>=1.1.30",
    "torch>=2.0.0",
    "torchvision>=0.15.0",
    "opencv-python>=4.8.0",
    "numpy>=1.24.0",
    "pandas>=2.0.0",
    "matplotlib>=3.7.0",
    "pyyaml>=6.0",
    "pillow>=9.5.0",
    "",
])
(EXPORT_DIR / "requirements.txt").write_text(REQUIREMENTS, encoding="utf-8")
print("collected into:", EXPORT_DIR)
for p in sorted(EXPORT_DIR.rglob("*")):
    if p.is_file():
        print("   ", p.relative_to(EXPORT_DIR))

In [ ]:
zip_base = str(PROJECT_DIR / "cv_project_submission")
archive = shutil.make_archive(zip_base, "zip", root_dir=str(EXPORT_DIR))
size_mb = Path(archive).stat().st_size / 1e6
print("archive: %s  (%.1f MB)" % (archive, size_mb))

try:
    from google.colab import files
    files.download(archive)
except Exception as e:
    print("(not running in Colab, or the download was blocked:", e, ")")
    print("The archive is on disk at the path above.")

In [ ]:
EXPORT_ONNX = False

if EXPORT_ONNX:
    try:
        m = YOLO(RUN_REGISTRY[BEST_TAG]["best"])
        onnx_path = m.export(format="onnx", imgsz=IMGSZ, opset=12, dynamic=False, simplify=True)
        shutil.copy2(onnx_path, EXPORT_DIR / "weights" / Path(onnx_path).name)
        print("exported:", onnx_path)
    except Exception as e:
        print("ONNX export failed:", type(e).__name__, e)